# Evaluation of IAF of hourly CPM emulators

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import dask
import dask.array
from dask.distributed import Client
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_analysis.iaf import calc_pmf, plot_pmf

In [ ]:
client = Client()
client

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.furflex_magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS["CPM"]

## Area df

In [ ]:
var = "pr"

In [ ]:
%%time

target_pmf = calc_pmf(TARGET_DAS[var])

pred_pmf = calc_pmf(PRED_DAS[var]).mean("sample_id")

In [ ]:
norm = matplotlib.colors.LogNorm(vmin=1e-5, vmax=0.5)
diff_norm = matplotlib.colors.SymLogNorm(linthresh=0.0001, vmin=-0.5, vmax=0.5)
rel_diff_norm = matplotlib.colors.Normalize(vmin=-500, vmax=500)

### Probabilities

In [ ]:
%%time

entries = ["target"] + list(pred_pmf["model"].values)
cols = 3
npads = (cols - len(entries) % cols) % cols
grid_spec = np.pad(np.array(entries), (0,npads), mode='constant', constant_values=".").reshape(-1, cols)

fig = plt.figure(layout="constrained", figsize=(2*grid_spec.shape[1]+1, 2*grid_spec.shape[0]))
axd = fig.subplot_mosaic(grid_spec, sharex=True, sharey=True)

ax = axd["target"]
shw = plot_pmf(ax, target_pmf, title="CPM", norm=norm)

for model, model_pmf in pred_pmf.groupby("model"):
    ax = axd[model]
    shw = plot_pmf(ax, model_pmf.squeeze(), title=model, norm=norm)

cb = fig.colorbar(
    shw,
    ax=axd.values(),
    location="right",
    extend="max",
)
cb.set_label("Probability mass", fontsize="small")
cb.ax.tick_params(labelsize="small")

fig.supxlabel("Maximum intensity (mm/hr)")
fig.supylabel("Cell area duration (# gridboxes)")

plt.show()

### Error

In [ ]:
%%time

entries = ["target"] + list(pred_pmf["model"].values)
cols = 3
npads = (cols - len(entries) % cols) % cols
grid_spec = np.pad(np.array(entries), (0,npads), mode='constant', constant_values=".").reshape(-1, cols)

fig = plt.figure(layout="constrained", figsize=(2*grid_spec.shape[1]+1, 2*grid_spec.shape[0]+0.5))
axd = fig.subplot_mosaic(grid_spec, sharex=True, sharey=True)

ax = axd["target"]
shw = plot_pmf(ax, target_pmf, title="CPM", norm=norm)
cb = fig.colorbar(
    shw,
    ax=ax,
    location="right",
    extend="max",
)
cb.set_label("Probability mass", fontsize="small")
cb.ax.tick_params(labelsize="x-small")

for model, model_pmf in pred_pmf.groupby("model"):
    ax = axd[model]
    shw = plot_pmf(ax, model_pmf.squeeze() - target_pmf, title=model, cmap="RdBu", norm=diff_norm)

cb = fig.colorbar(
    shw,
    ax=[ax for k,ax in axd.items() if k != "target"],
    location="right",
    extend="both",
)
cb.set_label("Probability mass error", fontsize="small")
cb.ax.tick_params(labelsize="small")

fig.supxlabel("Maximum intensity (mm/hr)")
fig.supylabel("Cell area(# gridboxes)")

plt.show()

### Relative error

In [ ]:
%%time

entries = ["target"] + list(pred_pmf["model"].values)
cols = 3
npads = (cols - len(entries) % cols) % cols
grid_spec = np.pad(np.array(entries), (0,npads), mode='constant', constant_values=".").reshape(-1, cols)

fig = plt.figure(layout="constrained", figsize=(2*grid_spec.shape[1]+1, 2*grid_spec.shape[0]+0.5))
axd = fig.subplot_mosaic(grid_spec, sharex=True, sharey=True)


ax = axd["target"]
shw = plot_pmf(ax, target_pmf, title="CPM", norm=norm)
cb = fig.colorbar(
    shw,
    ax=ax,
    location="right",
    extend="max",
)
cb.set_label("Probability mass", fontsize="small")
cb.ax.tick_params(labelsize="x-small")

for model, model_pmf in pred_pmf.groupby("model"):
    ax = axd[model]
    shw = plot_pmf(ax, 100*(model_pmf.squeeze() - target_pmf)/target_pmf, title=model, cmap="RdBu", norm=rel_diff_norm)

cb = fig.colorbar(
    shw,
    ax=[ax for k,ax in axd.items() if k != "target"],
    location="right",
    extend="both",
)
cb.set_label("Probability mass relative error", fontsize="small")
cb.ax.tick_params(labelsize="small")

fig.supxlabel("Maximum intensity (mm/hr)")
fig.supylabel("Cell area(# gridboxes)")

plt.show()

In [ ]:
client.close()